In [ ]:
%%writefile lab6.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>

/* Структура заявки */
typedef struct {
    int  id;
    char description[100];
    char status[50];
    char date[20];
} Request;

/* Узел дерева */
typedef struct Node {
    Request      data;
    struct Node *left;
    struct Node *right;
} Node;

/* Узел очереди для BFS */
typedef struct QNode {
    Node         *treeNode;
    struct QNode *next;
} QNode;

/* Очередь */
typedef struct {
    QNode *front;
    QNode *rear;
} Queue;

/* ---- Очередь ---- */

Queue *createQueue() {
    Queue *q = malloc(sizeof(Queue));
    q->front = q->rear = NULL;
    return q;
}

void enqueue(Queue *q, Node *n) {
    QNode *qn = malloc(sizeof(QNode));
    qn->treeNode = n;
    qn->next = NULL;
    if (q->rear) q->rear->next = qn;
    else q->front = qn;
    q->rear = qn;
}

Node *dequeue(Queue *q) {
    if (!q->front) return NULL;
    QNode *tmp = q->front;
    Node  *n   = tmp->treeNode;
    q->front   = tmp->next;
    if (!q->front) q->rear = NULL;
    free(tmp);
    return n;
}

int isQueueEmpty(Queue *q) {
    return q->front == NULL;
}

/* ---- Ввод / вывод ---- */

static void clean_newline(char *s) {
    size_t len = strlen(s);
    if (len > 0 && s[len-1] == '\n') s[len-1] = '\0';
}

static void read_line(const char *prompt, char *buf, int n) {
    printf("%s", prompt);
    fflush(stdout);
    if (fgets(buf, n, stdin)) clean_newline(buf);
    else buf[0] = '\0';
}

static void clear_buffer() {
    int c;
    while ((c = getchar()) != '\n' && c != EOF);
}

void readRequest(Request *r) {
    printf("Введите ID: ");
    scanf("%d", &r->id);
    clear_buffer();
    read_line("Описание проблемы: ",              r->description, 100);
    read_line("Статус (в обработке/завершено): ", r->status,       50);
    read_line("Дата подачи (ДД.ММ.ГГГГ): ",       r->date,         20);
}

void printRequest(const Request *r) {
    printf("  ID: %d | %s | %s | %s\n",
           r->id, r->description, r->status, r->date);
}

/* ---- Базовые операции дерева ---- */

Node *createNode(Request r) {
    Node *n  = malloc(sizeof(Node));
    n->data  = r;
    n->left  = n->right = NULL;
    return n;
}

Node *insert(Node *root, Request r) {
    if (!root) return createNode(r);
    if (r.id < root->data.id)
        root->left  = insert(root->left,  r);
    else if (r.id > root->data.id)
        root->right = insert(root->right, r);
    else
        printf("Заявка с ID=%d уже существует.\n", r.id);
    return root;
}

Node *search(Node *root, int id) {
    if (!root)               return NULL;
    if (id == root->data.id) return root;
    if (id <  root->data.id) return search(root->left,  id);
                              return search(root->right, id);
}

/* ---- Удаление узла (3 случая) ---- */

/* Вспомогательная: минимальный узел поддерева */
static Node *minNode(Node *root) {
    while (root->left) root = root->left;
    return root;
}

Node *deleteNode(Node *root, int id) {
    if (!root) {
        printf("Заявка с ID=%d не найдена.\n", id);
        return NULL;
    }
    if (id < root->data.id) {
        root->left  = deleteNode(root->left,  id);
    } else if (id > root->data.id) {
        root->right = deleteNode(root->right, id);
    } else {
        /* Случай 1: лист */
        if (!root->left && !root->right) {
            free(root);
            return NULL;
        }
        /* Случай 2: один потомок */
        if (!root->left) {
            Node *tmp = root->right;
            free(root);
            return tmp;
        }
        if (!root->right) {
            Node *tmp = root->left;
            free(root);
            return tmp;
        }
        /* Случай 3: два потомка — заменяем минимальным из правого поддерева */
        Node *successor  = minNode(root->right);
        root->data       = successor->data;
        root->right      = deleteNode(root->right, successor->data.id);
    }
    return root;
}

/* ---- Минимум и максимум ---- */

Node *findMin(Node *root) {
    if (!root) return NULL;
    while (root->left) root = root->left;
    return root;
}

Node *findMax(Node *root) {
    if (!root) return NULL;
    while (root->right) root = root->right;
    return root;
}

/* ---- Высота дерева ---- */

int height(Node *root) {
    if (!root) return 0;
    int lh = height(root->left);
    int rh = height(root->right);
    return 1 + (lh > rh ? lh : rh);
}

/* ---- Полное удаление дерева ---- */

void freeTree(Node *root) {
    if (!root) return;
    freeTree(root->left);
    freeTree(root->right);
    free(root);
}

/* ---- Обходы ---- */

void inorder(Node *root) {
    if (!root) return;
    inorder(root->left);
    printRequest(&root->data);
    inorder(root->right);
}

void bfs(Node *root) {
    if (!root) return;
    Queue *q = createQueue();
    enqueue(q, root);
    while (!isQueueEmpty(q)) {
        Node *cur = dequeue(q);
        printRequest(&cur->data);
        if (cur->left)  enqueue(q, cur->left);
        if (cur->right) enqueue(q, cur->right);
    }
    free(q);
}

/* ---- Сохранение и загрузка (текстовый формат) ---- */

/* Сохраняем inorder-обходом: при загрузке дерево будет сбалансировано */
static void saveInorder(Node *root, FILE *f) {
    if (!root) return;
    saveInorder(root->left, f);
    fprintf(f, "%d|%s|%s|%s\n",
            root->data.id,
            root->data.description,
            root->data.status,
            root->data.date);
    saveInorder(root->right, f);
}

void saveToFile(Node *root, const char *filename) {
    FILE *f = fopen(filename, "w");
    if (!f) { perror("saveToFile"); return; }
    /* Первая строка — высота (удобно для проверки при загрузке) */
    fprintf(f, "%d\n", height(root));
    saveInorder(root, f);
    fclose(f);
    printf("Дерево сохранено в файл '%s'.\n", filename);
}

Node *loadFromFile(const char *filename) {
    FILE *f = fopen(filename, "r");
    if (!f) { printf("Файл '%s' не найден.\n", filename); return NULL; }

    int h;
    fscanf(f, "%d\n", &h);  /* читаем высоту (не используем, просто пропускаем) */

    Node *root = NULL;
    char line[200];
    while (fgets(line, sizeof(line), f)) {
        clean_newline(line);
        if (strlen(line) == 0) continue;
        Request r;
        /* Парсим строку формата id|desc|status|date */
        char *tok = strtok(line, "|");
        if (!tok) continue;
        r.id = atoi(tok);
        tok = strtok(NULL, "|"); if (tok) strncpy(r.description, tok, 99);
        tok = strtok(NULL, "|"); if (tok) strncpy(r.status,      tok, 49);
        tok = strtok(NULL, "|"); if (tok) strncpy(r.date,        tok, 19);
        root = insert(root, r);
    }
    fclose(f);
    printf("Дерево загружено из файла '%s'.\n", filename);
    return root;
}

/* ---- Главное меню ---- */

int main() {
    Node *root = NULL;
    int choice, id;
    const char *filename = "requests_bst.txt";

    do {
        printf("\n1.  Добавить заявку\n");
        printf("2.  Удалить заявку по ID\n");
        printf("3.  Показать все (inorder / LNR)\n");
        printf("4.  Показать все (BFS)\n");
        printf("5.  Поиск по ID\n");
        printf("6.  Минимум и максимум\n");
        printf("7.  Высота дерева\n");
        printf("8.  Сохранить в файл\n");
        printf("9.  Загрузить из файла\n");
        printf("10. Удалить всё дерево\n");
        printf("0.  Выход\n");
        printf("Выбор: ");
        if (scanf("%d", &choice) != 1) break;
        clear_buffer();

        switch (choice) {
            case 1: {
                Request r;
                readRequest(&r);
                root = insert(root, r);
                printf("Заявка добавлена.\n");
                break;
            }
            case 2:
                printf("Введите ID для удаления: ");
                scanf("%d", &id); clear_buffer();
                root = deleteNode(root, id);
                if (root || id == 0)
                    printf("Операция удаления выполнена.\n");
                break;
            case 3:
                printf("\n--- Inorder (LNR) ---\n");
                if (!root) printf("Дерево пусто.\n");
                else inorder(root);
                break;
            case 4:
                printf("\n--- BFS ---\n");
                if (!root) printf("Дерево пусто.\n");
                else bfs(root);
                break;
            case 5:
                printf("Введите ID для поиска: ");
                scanf("%d", &id); clear_buffer();
                {
                    Node *found = search(root, id);
                    if (found) { printf("Найдено:\n"); printRequest(&found->data); }
                    else printf("Заявка с ID=%d не найдена.\n", id);
                }
                break;
            case 6:
                if (!root) { printf("Дерево пусто.\n"); break; }
                printf("Минимум: "); printRequest(&findMin(root)->data);
                printf("Максимум: "); printRequest(&findMax(root)->data);
                break;
            case 7:
                printf("Высота дерева: %d\n", height(root));
                break;
            case 8:
                saveToFile(root, filename);
                break;
            case 9:
                freeTree(root);
                root = loadFromFile(filename);
                break;
            case 10:
                freeTree(root);
                root = NULL;
                printf("Дерево полностью удалено из памяти.\n");
                break;
            case 0:
                break;
            default:
                printf("Неверный выбор.\n");
        }
    } while (choice != 0);

    freeTree(root);
    printf("Память освобождена. Выход.\n");
    return 0;
}

Writing lab6.c


In [ ]:
!gcc lab6.c -o lab6 -Wall
!./lab6


1.  Добавить заявку
2.  Удалить заявку по ID
3.  Показать все (inorder / LNR)
4.  Показать все (BFS)
5.  Поиск по ID
6.  Минимум и максимум
7.  Высота дерева
8.  Сохранить в файл
9.  Загрузить из файла
10. Удалить всё дерево
0.  Выход
Выбор: 1
Введите ID: 123
Описание проблемы: не работает мышка
Статус (в обработке/завершено): в обработке
Дата подачи (ДД.ММ.ГГГГ): 29.03.2026
Заявка добавлена.

1.  Добавить заявку
2.  Удалить заявку по ID
3.  Показать все (inorder / LNR)
4.  Показать все (BFS)
5.  Поиск по ID
6.  Минимум и максимум
7.  Высота дерева
8.  Сохранить в файл
9.  Загрузить из файла
10. Удалить всё дерево
0.  Выход
Выбор: 1
Введите ID: 456
Описание проблемы: переустановка видны
Статус (в обработке/завершено): завершено
Дата подачи (ДД.ММ.ГГГГ): 26.03.2026
Заявка добавлена.

1.  Добавить заявку
2.  Удалить заявку по ID
3.  Показать все (inorder / LNR)
4.  Показать все (BFS)
5.  Поиск по ID
6.  Минимум и максимум
7.  Высота дерева
8.  Сохранить в файл
9.  Загрузить из файла
10.